In [1]:
# 1: Mount Google Drive and load the JSON file, then check it has 1000 examples

from google.colab import drive
import json
import os

# Mount Drive
drive.mount('/content/drive')

# Path to the JSON file in Google Drive
json_path = "/content/drive/MyDrive/hotpotqa_data/hotpotqa_dev_2017wiki_1000.json"

# Load JSON
with open(json_path, "r", encoding="utf-8") as f:
    examples = json.load(f)

# Basic sanity checks
if not isinstance(examples, list):
    print(f"Unexpected type: {type(examples)} (expected list)")
else:
    num_examples = len(examples)
    print(f"Number of examples in JSON: {num_examples}")
    if num_examples == 1000:
        print("OK: JSON contains 1000 examples.")
    else:
        print("Warning: JSON does NOT contain 1000 examples.")

Mounted at /content/drive
Number of examples in JSON: 1000
OK: JSON contains 1000 examples.


In [2]:
# 2: Show  example to inspect the structure / keys

from pprint import pprint

# Pick a index
print(f"example index: {0}")

# Pretty-print the example
pprint(examples[0], width=120, sort_dicts=False)

example index: 0
{'_id': '5a8c7595554299585d9e36b6',
 'answer': 'Chief of Protocol',
 'question': 'What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?',
 'supporting_facts': [['Kiss and Tell (1945 film)', 0], ['Shirley Temple', 0], ['Shirley Temple', 1]],
 'context': [['Meet Corliss Archer',
              ["Meet Corliss Archer, a program from radio's Golden Age, ran from January 7, 1943 to September 30, "
               '1956.',
               ' Although it was CBS\'s answer to NBC\'s popular "A Date with Judy", it was also broadcast by NBC in '
               '1948 as a summer replacement for "The Bob Hope Show".',
               ' From October 3, 1952 to June 26, 1953, it aired on ABC, finally returning to CBS.',
               " Despite the program's long run, fewer than 24 episodes are known to exist."]],
             ['Shirley Temple',
              ['Shirley Temple Black (April 23, 1928 – February 10, 2014) was an American actres

In [3]:
# 3: Show one random example and compare context vs docs per title

import random
from pprint import pprint

# Pick a random example
idx = random.randrange(len(examples))  # 'examples' is loaded in the first cell
ex = examples[idx]

print(f"Random example index: {idx}")
print("=" * 80)
print(f"_id:      {ex.get('_id')}")
print(f"type:     {ex.get('type')}")
print(f"level:    {ex.get('level')}")
print(f"question: {ex.get('question')}")
print(f"answer:   {ex.get('answer')}")
print("=" * 80)

# Optionally show supporting facts
print("Supporting facts:")
pprint(ex.get("supporting_facts"), width=120, sort_dicts=False)
print("=" * 80)

# Build a mapping: context title -> joined paragraph
context_map = {}
for item in ex.get("context", []):
    # Each item should be [title, [sent_0, sent_1, ...]]
    if isinstance(item, (list, tuple)) and len(item) == 2:
        ctx_title, sent_list = item
        if isinstance(ctx_title, str) and isinstance(sent_list, list):
            paragraph = " ".join(
                s.strip() for s in sent_list
                if isinstance(s, str) and s.strip()
            )
            context_map[ctx_title] = paragraph

# Show aligned titles, context and docs
print("Titles, context paragraphs and docs (aligned by title):\n")

titles = ex.get("titles", [])
docs = ex.get("docs", [])
max_chars = 4000  # limit printed length for readability

for i, (t, d) in enumerate(zip(titles, docs)):
    print(f"[{i}] Title: {t}")

    # Context snippet for this title (show '\n' explicitly via repr)
    ctx_para = context_map.get(t)
    if ctx_para:
        ctx_snip = repr(ctx_para)
        if len(ctx_snip) > max_chars:
            ctx_snip = ctx_snip[:max_chars] + "..."
        print("    Context snippet:")
        print("    " + ctx_snip)
    else:
        print("    Context snippet: <no context paragraph for this title>")

    # Doc snippet for this title (show '\n' explicitly via repr)
    doc_snip = repr(d)
    if len(doc_snip) > max_chars:
        doc_snip = doc_snip[:max_chars] + "..."
    print("    Doc snippet:")
    print("    " + doc_snip)

    print("-" * 80)

Random example index: 973
_id:      5a77474855429972597f14e4
type:     comparison
level:    hard
question: Who released their debut album first, Daisy Chainsaw or Generationals?
answer:   Daisy Chainsaw
Supporting facts:
[['Daisy Chainsaw', 0], ['Daisy Chainsaw', 1], ['Generationals', 0], ['Generationals', 1]]
Titles, context paragraphs and docs (aligned by title):

[0] Title: Edward (EP)
    Context snippet:
    'Edward is an EP from London singer-songwriter Emma-Lee Moss, better known as Emmy the Great, released on August 10, 2009 on UK indie label Close Harbour Records. It is a collection of songs written before the release of her debut album First Love but not recorded in the album sessions. Moss stated on her MySpace page that after playing these songs on tour, she was reminded of the joys of song-writing, and inspired to record the songs for a spontaneous release.'
    Doc snippet:
    'Edward is an EP from London singer-songwriter Emma-Lee Moss, better known as Emmy the Great, r

In [4]:
# 4: Helper functions for docs_chunks, docs, title_chunks, and supports

from typing import List, Dict, Any
import re

def build_context_map(ex: Dict[str, Any]) -> Dict[str, List[str]]:
    # Map each title to its context sentences from "context"
    ctx: Dict[str, List[str]] = {}
    for item in ex.get("context", []):
        if isinstance(item, (list, tuple)) and len(item) == 2:
            title, sent_list = item
            if isinstance(title, str) and isinstance(sent_list, list):
                sentences = [
                    s.strip()
                    for s in sent_list
                    if isinstance(s, str) and s.strip()
                ]
                if title in ctx:
                    ctx[title].extend(sentences)
                else:
                    ctx[title] = sentences
    return ctx


def make_docs_chunks(ex: Dict[str, Any]) -> List[List[str]]:
    # For each title, collect its context sentences (or [] if none)
    ctx_map = build_context_map(ex)
    titles = ex.get("titles", [])
    return [list(ctx_map.get(t, [])) for t in titles]


def make_docs_with_paragraph_newlines(ex: Dict[str, Any]) -> List[str]:
    """
    Build docs so that:
      - Each doc remains a single string in the list.
      - Paragraph boundaries are kept and normalized.
      - Multiple newlines / blank lines are collapsed to a single '\n'.
      - There should be no '\n\n' sequences in the final docs.
    """
    docs_in = ex.get("docs", [])
    out_docs: List[str] = []

    for d in docs_in:
        if not isinstance(d, str):
            d = str(d)

        # Normalize different newline styles to Unix-style '\n'
        text = d.replace("\r\n", "\n").replace("\r", "\n")

        # Collapse multiple newlines (possibly with spaces) into a single '\n'
        # This removes blank lines and ensures paragraph breaks are exactly one '\n'
        text = re.sub(r"\n\s*\n+", "\n", text)

        # Strip leading and trailing whitespace (including newlines)
        text = text.strip()

        out_docs.append(text)

    return out_docs


def make_title_chunks(ex: Dict[str, Any]) -> List[List[str]]:
    # Build [title, sentence] pairs from docs2
    titles = ex.get("titles", [])
    docs2 = ex.get("docs2", [])
    out: List[List[str]] = []

    for t, sent_list in zip(titles, docs2):
        if isinstance(sent_list, list):
            for s in sent_list:
                if isinstance(s, str) and s.strip():
                    out.append([t, s.strip()])

    return out


def make_supports(ex: Dict[str, Any]) -> List[List[str]]:
    # Build [title, sentence] pairs from supporting_facts + context
    ctx_map = build_context_map(ex)
    out: List[List[str]] = []

    for title, idx in ex.get("supporting_facts", []):
        if not isinstance(title, str):
            continue
        sent_list = ctx_map.get(title, [])
        if isinstance(idx, int) and 0 <= idx < len(sent_list):
            sentence = sent_list[idx].strip()
            if sentence:
                out.append([title, sentence])

    return out

In [5]:
# 5: Build transformed examples with the desired structure (+ add "type" after "answer")

from typing import List, Dict, Any

new_examples: List[Dict[str, Any]] = []

for ex in examples:
    new_ex = {
        "question": ex.get("question"),
        "answer": ex.get("answer"),
        "type": ex.get("type"),
        "titles": list(ex.get("titles", [])),           # keep titles order
        "docs_chunks": make_docs_chunks(ex),            # sentence-level from context
        "docs": make_docs_with_paragraph_newlines(ex),  # docs with '\n' as paragraph separators
        "title_chunks": make_title_chunks(ex),          # [title, sentence] from docs2
        "supports": make_supports(ex),                  # [title, sentence] from supporting_facts
    }
    new_examples.append(new_ex)

print(f"Built {len(new_examples)} transformed examples.")

# Quick sanity check on the first example
ex0 = new_examples[0]
print("Keys:", list(ex0.keys()))
print("type:", ex0.get("type"))
print("len(titles):      ", len(ex0["titles"]))
print("len(docs_chunks): ", len(ex0["docs_chunks"]))
print("len(docs):        ", len(ex0["docs"]))
print("len(title_chunks):", len(ex0["title_chunks"]))
print("len(supports):    ", len(ex0["supports"]))


Built 1000 transformed examples.
Keys: ['question', 'answer', 'type', 'titles', 'docs_chunks', 'docs', 'title_chunks', 'supports']
type: bridge
len(titles):       10
len(docs_chunks):  10
len(docs):         10
len(title_chunks): 467
len(supports):     3


In [6]:
# 6: Save the transformed JSON into Google Drive under /content/drive/MyDrive/final_project
#    The file will be a proper JSON with double quotes (json.dump).

import os
import json

output_dir = "/content/drive/MyDrive/final_project"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "hotpotqa_dev_2017wiki_1000_converted.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(new_examples, f, ensure_ascii=False, indent=2)

print(f"Saved converted JSON to: {output_path}")

Saved converted JSON to: /content/drive/MyDrive/final_project/hotpotqa_dev_2017wiki_1000_converted.json


In [7]:
# 7: Show first example from the original JSON (as loaded at the start of this notebook)
#    We print a JSON-formatted string (no truncation).

import json

if not examples:
    print("No examples in original JSON.")
else:
    orig_ex0 = examples[0]
    print("=== Original JSON: examples[0] ===")
    print(json.dumps(orig_ex0, ensure_ascii=False, indent=2))

=== Original JSON: examples[0] ===
{
  "_id": "5a8c7595554299585d9e36b6",
  "answer": "Chief of Protocol",
  "question": "What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?",
  "supporting_facts": [
    [
      "Kiss and Tell (1945 film)",
      0
    ],
    [
      "Shirley Temple",
      0
    ],
    [
      "Shirley Temple",
      1
    ]
  ],
  "context": [
    [
      "Meet Corliss Archer",
      [
        "Meet Corliss Archer, a program from radio's Golden Age, ran from January 7, 1943 to September 30, 1956.",
        " Although it was CBS's answer to NBC's popular \"A Date with Judy\", it was also broadcast by NBC in 1948 as a summer replacement for \"The Bob Hope Show\".",
        " From October 3, 1952 to June 26, 1953, it aired on ABC, finally returning to CBS.",
        " Despite the program's long run, fewer than 24 episodes are known to exist."
      ]
    ],
    [
      "Shirley Temple",
      [
        "Shirley Temple B

In [8]:
# 8: Show first example from the converted JSON we just wrote
#    This should match the target structure you described.

import json

conv_path = "/content/drive/MyDrive/final_project/hotpotqa_dev_2017wiki_1000_converted.json"

with open(conv_path, "r", encoding="utf-8") as f:
    converted = json.load(f)

print(f"Number of converted examples: {len(converted)}")

conv_ex0 = converted[0]
print("=== Converted JSON: examples[0] ===")
print(json.dumps(conv_ex0, ensure_ascii=False, indent=2))

Number of converted examples: 1000
=== Converted JSON: examples[0] ===
{
  "question": "What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?",
  "answer": "Chief of Protocol",
  "type": "bridge",
  "titles": [
    "Meet Corliss Archer",
    "Shirley Temple",
    "Janet Waldo",
    "Meet Corliss Archer (TV series)",
    "Lord High Treasurer",
    "A Kiss for Corliss",
    "Kiss and Tell (1945 film)",
    "Secretary of State for Constitutional Affairs",
    "Village accountant",
    "Charles Craft"
  ],
  "docs_chunks": [
    [
      "Meet Corliss Archer, a program from radio's Golden Age, ran from January 7, 1943 to September 30, 1956.",
      "Although it was CBS's answer to NBC's popular \"A Date with Judy\", it was also broadcast by NBC in 1948 as a summer replacement for \"The Bob Hope Show\".",
      "From October 3, 1952 to June 26, 1953, it aired on ABC, finally returning to CBS.",
      "Despite the program's long run, fewer than